# PDD Mobility Scanner — Format trail_data.csv with YOLO Annotations

Combines two pipelines:

1. **Format conversion** (from `format_trail_data.ipynb`) — slices `trail_data.csv` into 5-second waypoints, extracts the first frame of each waypoint as `img_NNNN.jpg`, and produces the dashboard-compatible xlsx with `trail_data` + `waypoint_photos` sheets.
2. **YOLO segmentation** (from `yolo_video_inference.ipynb`) — runs a `.pt` segmentation model with ByteTrack, suppresses overlapping classes by mask-IoU, then aggregates detections per 5-second waypoint to fill in `surface_type`, `obstacles`, and `infrastructure`.

The original `format_trail_data.ipynb` still works without a model — use this notebook when you have a YOLO `.pt` and want auto-annotated output.

## Step 1: Install dependencies

In [ ]:
!pip install -q ultralytics

import pandas as pd
import numpy as np
import cv2
import os
import shutil
import torch
from collections import Counter
from ultralytics import YOLO
from google.colab import files

## Step 2: Upload files

Upload all four: `trail_data.csv`, `trail_video.mp4`, `video_sync.csv`, and your YOLO segmentation model `.pt`.

In [ ]:
uploaded = files.upload()

csv_files = [f for f in uploaded.keys() if f.endswith('.csv')]
trail_csv  = next(f for f in csv_files if 'sync' not in f.lower())
sync_csv   = next(f for f in csv_files if 'sync' in f.lower())
video_file = next(f for f in uploaded.keys() if f.lower().endswith('.mp4'))
model_path = next(f for f in uploaded.keys() if f.endswith('.pt'))

print(f'Trail CSV: {trail_csv}')
print(f'Sync CSV:  {sync_csv}')
print(f'Video:     {video_file}')
print(f'Model:     {model_path}')

## Step 3: Load sync info and sensor data

In [ ]:
sync = pd.read_csv(sync_csv)
rec_start_ms = int(sync['rec_start_ms'].iloc[0])
print(f'rec_start_ms = {rec_start_ms}')

df = pd.read_csv(trail_csv)
df['video_sec'] = (df['ms'] - rec_start_ms) / 1000.0
print(f'Loaded {len(df)} sensor rows, video duration {df["video_sec"].max():.1f}s')

## Step 4: Assign waypoints (5-second windows)

Pre-recording sensor rows fold into waypoint 0.

In [ ]:
WAYPOINT_SEC = 5
df['wp'] = np.maximum(0, np.floor(df['video_sec'] / WAYPOINT_SEC).astype(int))
n_waypoints = df['wp'].nunique()
print(f'{n_waypoints} waypoints over {len(df)} rows '
      f'(avg {len(df) / n_waypoints:.0f} rows per waypoint)')

## Step 5: Extract first frame per waypoint

Saved to `output/images/img_NNNN.jpg`.

In [ ]:
out_dir = 'output'
img_dir = os.path.join(out_dir, 'images')
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
os.makedirs(img_dir)

cap = cv2.VideoCapture(video_file)
fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {total_frames} frames at {fps:.2f} FPS')

wp_to_image = {}
for wp in sorted(df['wp'].unique()):
    target_sec = wp * WAYPOINT_SEC
    target_frame = min(int(round(target_sec * fps)), total_frames - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
    ret, frame = cap.read()
    if not ret:
        wp_to_image[wp] = wp_to_image.get(wp - 1, '')
        continue
    fname = f'img_{wp:04d}.jpg'
    cv2.imwrite(os.path.join(img_dir, fname), frame)
    wp_to_image[wp] = fname

cap.release()
print(f'Wrote {len([v for v in wp_to_image.values() if v])} images to {img_dir}')

## Step 6: Run YOLO segmentation tracking on every frame

Per-class confidence thresholds and overlap-suppression rules ported from `yolo_video_inference.ipynb`. ByteTrack persistence keeps `track_id` stable across frames so we can dedupe obstacles within a waypoint window.

In [ ]:
# --- Tunables ----------------------------------------------------------------
CLASS_CONF = {
    'bench': 0.15,
    'rough trail': 0.15,
}
DEFAULT_CONF = 0.35

SUPPRESS_RULES = [
    ('crushed gravel trail', 'rough trail'),  # rough trail wins on overlap
]
IOU_SUPPRESS = 0.3
USE_HALF = torch.cuda.is_available()
# -----------------------------------------------------------------------------

model = YOLO(model_path)
print('Model class names:', model.names)
print(f'FP16 inference: {USE_HALF}')
name_to_id = {v.lower(): k for k, v in model.names.items()}

def conf_for(class_id):
    return CLASS_CONF.get(model.names.get(class_id, '').lower(), DEFAULT_CONF)

def iou_mask(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0

GLOBAL_CONF = min([DEFAULT_CONF] + list(CLASS_CONF.values()))

cap = cv2.VideoCapture(video_file)
rows = []
frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    results = model.track(frame, persist=True, tracker='bytetrack.yaml',
                          conf=GLOBAL_CONF, half=USE_HALF, verbose=False)[0]

    if results.boxes is not None and len(results.boxes) > 0:
        confs = results.boxes.conf.cpu().numpy()
        clss  = results.boxes.cls.cpu().numpy().astype(int)
        masks_np = (results.masks.data.cpu().numpy().astype(bool)
                    if results.masks is not None else None)

        keep = [i for i, (c, k) in enumerate(zip(confs, clss)) if c >= conf_for(int(k))]

        suppressed = set()
        if masks_np is not None:
            for weaker_name, stronger_name in SUPPRESS_RULES:
                w_id = name_to_id.get(weaker_name.lower())
                s_id = name_to_id.get(stronger_name.lower())
                if w_id is None or s_id is None:
                    continue
                for i in keep:
                    if int(clss[i]) != w_id:
                        continue
                    for j in keep:
                        if int(clss[j]) != s_id:
                            continue
                        if iou_mask(masks_np[i], masks_np[j]) > IOU_SUPPRESS:
                            suppressed.add(i)
                            break
        keep = [i for i in keep if i not in suppressed]

        if results.boxes.id is not None:
            ids = results.boxes.id.cpu().numpy().astype(int)
        else:
            ids = np.full(len(results.boxes), -1, dtype=int)
        areas = (masks_np.sum(axis=(1, 2)) if masks_np is not None
                 else np.zeros(len(results.boxes), dtype=int))
        for i in keep:
            rows.append({
                'frame': frame_idx,
                'track_id': int(ids[i]),
                'class_id': int(clss[i]),
                'class_name': model.names.get(int(clss[i]), str(clss[i])),
                'conf': float(confs[i]),
                'mask_area': int(areas[i]),
            })

    frame_idx += 1
    if frame_idx % 25 == 0:
        print(f'  processed {frame_idx}/{total_frames} frames')

cap.release()
det = pd.DataFrame(rows)
n_tracks = det['track_id'].nunique() if len(det) else 0
print(f'\nDone. {frame_idx} frames processed, {len(det)} detections, {n_tracks} unique tracks.')
det.head()

## Step 7: Aggregate detections per 5-second waypoint

Each waypoint covers 5 video frames at 1 fps. For each waypoint:

- **`surface_type`** — sum mask area per trail class across the 5 frames; pick the trail class with the largest total area. If no trail class detected, fall back to `No surface detected`.
- **`obstacles`** / **`infrastructure`** — bucket detections by token-substring match against class names, then dedupe by `track_id` within the waypoint so a single physical object isn't double-counted across frames.

In [ ]:
# --- Tunables ----------------------------------------------------------------
TRAIL_CLASSES   = ['rough trail', 'crushed gravel trail', 'wood trail']
OBSTACLE_TOKENS = ['log on ground', 'rock', 'root', 'branch', 'debris', 'hole', 'puddle']
INFRA_TOKENS    = ['sign', 'bench', 'boardwalk', 'railing', 'fence', 'post', 'bridge', 'trail blaze']
# -----------------------------------------------------------------------------

trail_set = {c.lower() for c in TRAIL_CLASSES}

def matches_tokens(class_name, tokens):
    cn = str(class_name).lower().strip()
    return any(t in cn for t in tokens)

# Map every detection's frame to a waypoint using the same rule as Step 4
if len(det):
    det = det.copy()
    det['wp'] = np.floor(det['frame'] / (WAYPOINT_SEC * fps)).astype(int)

wp_annotations = {}
for wp in sorted(df['wp'].unique()):
    wp_dets = det[det['wp'] == wp] if len(det) else pd.DataFrame()

    # surface_type: trail class with largest total mask area
    if len(wp_dets):
        trail_dets = wp_dets[wp_dets['class_name'].str.lower().isin(trail_set)]
        if len(trail_dets):
            area_by_class = trail_dets.groupby('class_name')['mask_area'].sum()
            surface_type  = area_by_class.idxmax()
        else:
            surface_type = 'No surface detected'
    else:
        surface_type = 'No surface detected'

    # obstacles / infrastructure: dedupe by track_id within the waypoint
    obstacles_list, infra_list = [], []
    obs_seen, inf_seen = set(), set()
    for _, d in wp_dets.iterrows():
        cn  = str(d['class_name'])
        tid = int(d['track_id'])
        key = tid if tid >= 0 else cn.lower().strip()
        if matches_tokens(cn, OBSTACLE_TOKENS) and key not in obs_seen:
            obs_seen.add(key)
            obstacles_list.append(cn)
        if matches_tokens(cn, INFRA_TOKENS) and key not in inf_seen:
            inf_seen.add(key)
            infra_list.append(cn)

    wp_annotations[int(wp)] = {
        'surface_type':   surface_type,
        'obstacles':      '; '.join(obstacles_list) if obstacles_list else 'None detected',
        'infrastructure': '; '.join(infra_list)     if infra_list     else 'None detected',
    }

print(f'Annotated {len(wp_annotations)} waypoints')
# Class-bucket report
if len(det):
    print('\nDetection class -> bucket:')
    for cn in sorted(det['class_name'].unique()):
        buckets = []
        if cn.lower() in trail_set:           buckets.append('surface')
        if matches_tokens(cn, OBSTACLE_TOKENS): buckets.append('obstacle')
        if matches_tokens(cn, INFRA_TOKENS):    buckets.append('infrastructure')
        if not buckets:                        buckets = ['unmatched']
        print(f'  {cn!r}: {"+".join(buckets)}')

## Step 8: Build the trail_data and waypoint_photos DataFrames

In [ ]:
df['image_file']   = df['wp'].map(wp_to_image)
df['surface_type'] = df['wp'].map(lambda wp: wp_annotations.get(int(wp), {}).get('surface_type', 'No surface detected'))
df['obstacle']     = df['wp'].map(lambda wp: wp_annotations.get(int(wp), {}).get('obstacles',    'None detected'))
df['substructure'] = df['wp'].map(lambda wp: wp_annotations.get(int(wp), {}).get('infrastructure','None detected'))

TARGET_COLUMNS = [
    'ms', 'utc', 'wp',
    'ax', 'ay', 'az',
    'gvx', 'gvy', 'gvz',
    'gx', 'gy', 'gz',
    'qw', 'qx', 'qy', 'qz',
    'mx', 'my', 'mz',
    'tof_mm', 'tof_status',
    'lat', 'lng', 'alt', 'speed', 'hdg', 'sats', 'hdop',
    'cal_sys', 'cal_gyro', 'cal_accel', 'cal_mag',
    'image_file', 'surface_type', 'obstacle', 'substructure',
]

missing = [c for c in TARGET_COLUMNS if c not in df.columns]
assert not missing, f'Missing columns from input: {missing}'

out_df = df[TARGET_COLUMNS]

waypoint_photos_df = (
    out_df[['wp', 'image_file']]
    .drop_duplicates(subset='wp')
    .sort_values('wp')
    .reset_index(drop=True)
)
waypoint_photos_df['surface_type']   = waypoint_photos_df['wp'].map(lambda wp: wp_annotations.get(int(wp), {}).get('surface_type',   'No surface detected'))
waypoint_photos_df['obstacles']      = waypoint_photos_df['wp'].map(lambda wp: wp_annotations.get(int(wp), {}).get('obstacles',      'None detected'))
waypoint_photos_df['infrastructure'] = waypoint_photos_df['wp'].map(lambda wp: wp_annotations.get(int(wp), {}).get('infrastructure', 'None detected'))

print(f'trail_data:      {len(out_df)} rows, {len(out_df.columns)} columns')
print(f'waypoint_photos: {len(waypoint_photos_df)} rows')
waypoint_photos_df.head(10)

## Step 9: Write XLSX and download bundle

In [ ]:
xlsx_path = os.path.join(out_dir, 'trail_data.xlsx')
with pd.ExcelWriter(xlsx_path) as writer:
    out_df.to_excel(writer, sheet_name='trail_data', index=False)
    waypoint_photos_df.to_excel(writer, sheet_name='waypoint_photos', index=False)
print(f'Wrote {xlsx_path} (sheets: trail_data, waypoint_photos)')

bundle = shutil.make_archive('trail_data_annotated', 'zip', out_dir)
print(f'Bundle: {bundle}')
files.download(bundle)